In [1]:
!pip install -q transformers accelerate bitsandbytes pdfplumber gradio

In [2]:
!pip install --upgrade Pillow

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline

model_id = "microsoft/Phi-3-mini-4k-instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.padding_side = 'left'

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=quantization_config
)

llm_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

print("\n Phase 2 Completed")

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]


 Phase 2 Completed


In [4]:
# System check
messages = [
    {"role": "system", "content": "You are a strict academic tutor."},
    {"role": "user", "content": "Explain what a multi-agent system is in exactly one sentence."}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
outputs = llm_pipeline(prompt, max_new_tokens=100, do_sample=True, temperature=0.3)
print(outputs[0]['generated_text'][len(prompt):].strip())

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destru

A multi-agent system is a computational framework where multiple autonomous agents interact and collaborate to achieve individual or collective objectives.


Agent 1 - Pdf Reader


In [5]:
import pdfplumber
import re

In [6]:
def pdf_reader_agent(file_path, max_chars=12000):

    print(f"PDF Reader Agent: Processing '{file_path}'")
    extracted_text = ""

    try:
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                text = page.extract_text()
                if text:
                    extracted_text += text + "\n"

        # Clean up excessive newlines and spaces common in PDFs
        clean_text = re.sub(r'\s+', ' ', extracted_text).strip()

        # Enforce context window limit
        if len(clean_text) > max_chars:
            print(f"PDF Reader Agent: Document too large. Truncating to first {max_chars} characters.")
            clean_text = clean_text[:max_chars]

        print(f"PDF Reader Agent: Extracted {len(clean_text)} characters.")
        return clean_text

    except Exception as e:
        return f"Error reading PDF: {str(e)}"

Agent 2 - Summary Provider


In [7]:
def summary_agent(text_content):
    """
    Takes extracted text and uses the LLM to generate bulleted revision notes.
    """
    if text_content.startswith("Error"):
        return "Cannot summarize due to file read error."

    print("Summary Agent: Analyzing material and generating summary...")

    # 1. Define the System Prompt (The Agent's Persona & Rules)
    messages = [
        {"role": "system", "content": "You are an expert academic study assistant. Your task is to extract the core concepts, definitions, and main arguments from the provided text. Present them strictly as a clear, concise bulleted summary."},
        {"role": "user", "content": f"Please summarize these lecture notes:\n\n{text_content}"}
    ]

    # 2. Format the prompt for the Phi-3 architecture
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # 3. Generate the response
    outputs = llm_pipeline(
        prompt,
        max_new_tokens=500,     # Allow enough length for a solid summary
        do_sample=True,
        temperature=0.2,        # Low temperature keeps the model factual and less "creative"
        return_full_text=False  # Tells pipeline to only return the generated answer, not the prompt
    )

    # Extract the generated string
    summary = outputs[0]['generated_text']
    print("Summary Agent: Summary complete.\n")
    return summary.strip()

Testing our both agents

In [9]:
test_pdf_path = "Updated_PoA_SoS'26 Chirag.pdf"

# Run Agent 1
pdf_content = pdf_reader_agent(test_pdf_path)

# Run Agent 2
if pdf_content and not pdf_content.startswith("Error"):
    final_summary = summary_agent(pdf_content)

    print(" FINAL OUTPUT:\n")
    print(final_summary)
else:
    print(pdf_content)

[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PDF Reader Agent: Processing 'Updated_PoA_SoS'26 Chirag.pdf'
PDF Reader Agent: Extracted 2768 characters.
Summary Agent: Analyzing material and generating summary...


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Summary Agent: Summary complete.

 FINAL OUTPUT:

- Week 1: Set up Python and PyTorch environment, review key concepts from Linear Algebra (vector spaces, eigenvalues) and Probability (Bayes' theorem, Gaussian distributions, KL divergence), study Maximum Likelihood Estimation, and get familiar with CNNs (convolutional layers, pooling, backpropagation).

- Week 2: Study Autoencoders and their bottleneck representations, move on to Variational Autoencoders (VAEs), understand the ELBO objective, implement a VAE on MNIST, and visualize the learned latent space.

- Week 3: Learn the theory of Generative Adversarial Networks (GANs), understand the min-max objective, discriminator-generator training dynamics, mode collapse, and training instability. Implement a DCGAN on CelebA.

- Week 4: Buffer week for cleanup, revisit difficult concepts from VAEs or GANs, draft and submit a midterm report on theoretical understanding and experimental findings.

- Week 5: Study score-based generative models

Agent 3 - Quiz Generator


In [10]:
def quiz_generator_agent(text_content):

    if not text_content or text_content.startswith("Error"):
        return "Cannot generate quiz: invalid text."

    print(" Quiz Agent: Crafting multiple-choice questions...")

    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert educator. Generate exactly 3 multiple-choice questions "
                "based strictly on the provided text. Format each question clearly with 4 options (A, B, C, D) "
                "and explicitly state the correct answer below the options."
            )
        },
        {"role": "user", "content": f"Create a quiz based on this text:\n\n{text_content}"}
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    outputs = llm_pipeline(
        prompt,
        max_new_tokens=600,     # Quizzes take up more tokens due to formatting
        do_sample=True,
        temperature=0.2,        # Keep it low so it doesn't hallucinate fake information
        return_full_text=False
    )

    print("Quiz Agent: Questions generated.\n")
    return outputs[0]['generated_text'].strip()

Agent 4 - Flashcard Generator

In [12]:
def flashcard_generator_agent(text_content):

    if not text_content or text_content.startswith("Error"):
        return "Cannot generate flashcards: invalid text."

    print("📇 Flashcard Agent: Extracting key terms...")

    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert study aid creator. Extract 5 key terms or concepts from the provided text "
                "and provide a concise definition for each. Format them strictly like this:\n\n"
                "Front: [Term]\nBack: [Definition]\n---"
            )
        },
        {"role": "user", "content": f"Create flashcards from this text:\n\n{text_content}"}
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    outputs = llm_pipeline(
        prompt,
        max_new_tokens=400,
        do_sample=True,
        temperature=0.1,        # Lowest temperature for maximum extraction accuracy
        return_full_text=False
    )

    print(" Flashcard Agent: Flashcards ready.\n")
    return outputs[0]['generated_text'].strip()

Testing our agents

In [13]:
test_text = """
Virat Kohli[b] (born 5 November 1988) is an Indian international cricketer and the former all-format captain of the Indian national cricket team. He is a right-handed batter and occasional right-arm medium-pace bowler. Considered one of the greatest batsmen in limited overs cricket, he has been acclaimed for his batting skills and records. Kohli has the most centuries in ODIs and the second-most centuries in international cricket with 85 tons across all formats. He is also the leading run-scorer in the Indian Premier League.
"""

quiz_output = quiz_generator_agent(test_text)
print(quiz_output)


flashcard_output = flashcard_generator_agent(test_text)
print(flashcard_output)

[transformers] Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 Quiz Agent: Crafting multiple-choice questions...


[transformers] Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Quiz Agent: Questions generated.

1. What is Virat Kohli' Premier League?

   A. A football league

   B. A cricket league

   C. A basketball league

   D. A hockey league

   Correct Answer: B. A cricket league


2. How many centuries has Virat Kohli scored in ODIs?

   A. 75

   B. 85

   C. 95

   D. 105

   Correct Answer: B. 85


3. What role did Virat Kohli serve in the Indian national cricket team?

   A. Wicketkeeper

   B. All-format captain

   C. Coach

   D. Team manager

   Correct Answer: B. All-format captain
📇 Flashcard Agent: Extracting key terms...
 Flashcard Agent: Flashcards ready.

Front: Virat Kohli
Back: An Indian international cricketer and former all-format captain of the Indian national cricket team, known for his exceptional batting skills and records in limited overs cricket.

Front: Limited Overs Cricket
Back: A form of cricket played with a set number of overs per team, typically 20 or 50, emphasizing batting and scoring runs.

Front: Centuries
Back: A te

Agent 5 - Doubt Solver

In [14]:
def doubt_solver_agent(text_content, user_question):

    if not text_content or text_content.startswith("Error"):
        return "Cannot answer: invalid text context."

    print(f" Doubt Solver: Searching text for answer to '{user_question}'...")

    messages = [
        {
            "role": "system",
            "content": (
                "You are an exact and honest academic tutor. Answer the user's question using ONLY "
                "the information provided in the text below. If the answer is not contained in the text, "
                "you must reply: 'I cannot find the answer to this in the provided notes.'"
            )
        },
        {
            "role": "user",
            "content": f"Text Context:\n{text_content}\n\nQuestion: {user_question}"
        }
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    outputs = llm_pipeline(
        prompt,
        max_new_tokens=300,
        do_sample=True,
        temperature=0.1,        # Very low temperature to prevent making things up
        return_full_text=False
    )

    return outputs[0]['generated_text'].strip()

Agent 6 - Study Planner

In [15]:
def study_planner_agent(text_content):

    if not text_content or text_content.startswith("Error"):
        return "Cannot generate plan: invalid text."

    print("Study Planner: Designing revision schedule...")

    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert academic strategist. Review the provided text and create a logical, "
                "3-step revision plan to master the material. Break it down by concepts."
            )
        },
        {"role": "user", "content": f"Create a study plan for these notes:\n\n{text_content}"}
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    outputs = llm_pipeline(
        prompt,
        max_new_tokens=400,
        do_sample=True,
        temperature=0.3,
        return_full_text=False
    )

    print("✅ Study Planner: Schedule ready.\n")
    return outputs[0]['generated_text'].strip()

Supervisor Architecture

In [16]:
class StudySupervisor:
    def __init__(self):
        # The supervisor remembers the text so agents can share the same context
        self.current_document_text = ""
        self.is_loaded = False

    def process_new_document(self, file_path):
        """Step 1: Read the PDF and store it in memory."""
        print("\n[Supervisor] Initiating document processing workflow...")
        text = pdf_reader_agent(file_path)

        if text.startswith("Error"):
            self.is_loaded = False
            return text

        self.current_document_text = text
        self.is_loaded = True
        return f"Document loaded successfully! Extracted {len(text)} characters."

    def run_agent(self, agent_name, *args):
        """Step 2: Route tasks to the appropriate specialized agent."""
        if not self.is_loaded:
            return "Please upload and process a document first."

        if agent_name == "summary":
            return summary_agent(self.current_document_text)
        elif agent_name == "quiz":
            return quiz_generator_agent(self.current_document_text)
        elif agent_name == "flashcards":
            return flashcard_generator_agent(self.current_document_text)
        elif agent_name == "planner":
            return study_planner_agent(self.current_document_text)
        elif agent_name == "doubt":
            # Doubt solver requires a second argument: the user's question
            question = args[0]
            return doubt_solver_agent(self.current_document_text, question)
        else:
            return "Unknown agent requested."

# Initialize our master controller
assistant = StudySupervisor()

Testing full model

In [19]:
# 1. Manually load our previous test text into the Supervisor's memory
assistant.current_document_text = """
Virat Kohli[b] (born 5 November 1988) is an Indian international cricketer and the former all-format captain of the Indian national cricket team. He is a right-handed batter and occasional right-arm medium-pace bowler. Considered one of the greatest batsmen in limited overs cricket, he has been acclaimed for his batting skills and records. Kohli has the most centuries in ODIs and the second-most centuries in international cricket with 85 tons across all formats. He is also the leading run-scorer in the Indian Premier League.
"""
assistant.is_loaded = True

# 2. Ask the Supervisor to run the Doubt Solver Agent
test_question = "When was Virat Kohli born?"
print(f"User asks: {test_question}\n")

answer = assistant.run_agent("doubt", test_question)
print(f"Output:\n{answer}\n")

# 3. Ask the Supervisor to run the Study Planner Agent
plan = assistant.run_agent("planner")
print(f"Output:\n{plan}")

[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


User asks: When was Virat Kohli born?

 Doubt Solver: Searching text for answer to 'When was Virat Kohli born?'...


[transformers] Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Output:
Virat Kohli was born on 5 November 1988.

Study Planner: Designing revision schedule...
✅ Study Planner: Schedule ready.

Output:
**Study Plan for Mastering Virat Kohli'ISBN**


**Step 1: Understanding the Concept**

- Read the provided text to gain a basic understanding of Virat Kohli's career and achievements.

- Identify key terms such as "right-handed batter," "right-arm medium-pace bowler," "limited overs cricket," "batting skills," "records," "centuries," "ODIs," "international cricket," and "Indian Premier League."

- Create a glossary of these key terms to refer back to during the revision process.


**Step 2: Deepening the Knowledge**

- Research the history of limited overs cricket and the significance of the ODI format.

- Study the career statistics of Virat Kohli, focusing on his batting records, centuries scored, and his role in the Indian national cricket team.

- Compare and contrast Virat Kohli's achievements with other cricketers to understand his place in the

Deployment

In [20]:
import gradio as gr


# 1. Wrapper functions to connect Gradio buttons to our Supervisor
def process_file(pdf_file):
    if pdf_file is None:
        return "⚠️ Please upload a PDF file first."
    # Gradio passes a file object; we extract the file path to give to our reader
    return assistant.process_new_document(pdf_file.name)

def get_summary(): return assistant.run_agent("summary")
def get_quiz(): return assistant.run_agent("quiz")
def get_flashcards(): return assistant.run_agent("flashcards")
def get_planner(): return assistant.run_agent("planner")
def solve_doubt(question):
    if not question.strip():
        return "Please type a question."
    return assistant.run_agent("doubt", question)

# 2. Build the UI Layout
with gr.Blocks(theme=gr.themes.Soft(), title="Multi-Agent Study Assistant") as demo:
    gr.Markdown("#  Multi-Agent Student Study Assistant")
    gr.Markdown("Upload your lecture notes and let specialized AI agents help you study.")

    with gr.Row():
        # LEFT COLUMN: File Upload and Status
        with gr.Column(scale=1):
            gr.Markdown("### 1. Upload Material")
            file_input = gr.File(label="Upload PDF Notes", file_types=[".pdf"])
            upload_btn = gr.Button("Process Document", variant="primary")
            upload_status = gr.Textbox(label="System Status", interactive=False)

        # RIGHT COLUMN: The Agent Workspaces (Organized in Tabs)
        with gr.Column(scale=2):
            gr.Markdown("### 2. Choose Your Agent")
            with gr.Tabs():

                # Tab 1: Summary Agent
                with gr.Tab("📝 Summary Agent"):
                    sum_btn = gr.Button("Generate Revision Notes")
                    sum_out = gr.Textbox(label="Agent Output", lines=12, interactive=False)

                # Tab 2: Quiz Generator
                with gr.Tab("❓ Quiz Generator"):
                    quiz_btn = gr.Button("Generate Practice Quiz")
                    quiz_out = gr.Textbox(label="Agent Output", lines=12, interactive=False)

                # Tab 3: Flashcard Generator
                with gr.Tab("📇 Flashcard Maker"):
                    flash_btn = gr.Button("Generate Key Term Flashcards")
                    flash_out = gr.Textbox(label="Agent Output", lines=12, interactive=False)

                # Tab 4: Study Planner
                with gr.Tab("📅 Study Planner"):
                    plan_btn = gr.Button("Generate Study Schedule")
                    plan_out = gr.Textbox(label="Agent Output", lines=12, interactive=False)

                # Tab 5: Doubt Solver
                with gr.Tab("🕵️ Doubt Solver"):
                    gr.Markdown("Ask a specific question about the uploaded material. The agent will only use the text to answer.")
                    doubt_input = gr.Textbox(label="Your Question", placeholder="e.g., What is the time complexity?")
                    doubt_btn = gr.Button("Ask Agent")
                    doubt_out = gr.Textbox(label="Agent Output", lines=5, interactive=False)

    # 3. Wire the buttons to the wrapper functions
    upload_btn.click(fn=process_file, inputs=file_input, outputs=upload_status)
    sum_btn.click(fn=get_summary, inputs=None, outputs=sum_out)
    quiz_btn.click(fn=get_quiz, inputs=None, outputs=quiz_out)
    flash_btn.click(fn=get_flashcards, inputs=None, outputs=flash_out)
    plan_btn.click(fn=get_planner, inputs=None, outputs=plan_out)
    doubt_btn.click(fn=solve_doubt, inputs=doubt_input, outputs=doubt_out)

# 4. Launch the application!
# share=True creates a temporary public URL so you can view it full-screen
demo.launch(debug=True, share=True)

/tmp/ipykernel_1631/1204171431.py:21: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Multi-Agent Study Assistant") as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c75bbb0641d328412f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)



[Supervisor] Initiating document processing workflow...
PDF Reader Agent: Processing '/tmp/gradio/abe16073dbd9570a1bd8b40639893c1ba07f930af5d32d60c2640e60cb8a1dc2/Hostel_Rules.pdf'
PDF Reader Agent: Extracted 6654 characters.


/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Summary Agent: Analyzing material and generating summary...


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Summary Agent: Summary complete.

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://c75bbb0641d328412f.gradio.live
